# 01 · Calidad de datos

Este notebook evalúa de forma reproducible la calidad de los datos de entrenamiento de **CommonLit - Evaluate Student Summaries**. Se revisan esquema, valores faltantes, duplicados, unicidad de identificadores, integridad referencial, texto, variables objetivo y distribución por prompt.

Las decisiones de limpieza son conservadoras: nunca se modifica `data/raw/`, no se imputan valores que no faltan y no se eliminan observaciones sin evidencia de error.

## 1. Preparación y carga

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Ejecute el notebook dentro del repositorio."
    )


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_training_data

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

summaries, prompts = load_training_data(RAW_DATA_DIR)
print(f"summaries_train: {summaries.shape[0]:,} filas × {summaries.shape[1]} columnas")
print(f"prompts_train:   {prompts.shape[0]:,} filas × {prompts.shape[1]} columnas")

summaries_train: 7,165 filas × 5 columnas
prompts_train:   4 filas × 4 columnas


## 2. Esquema y tipos de datos

In [2]:
EXPECTED_SCHEMA = {
    "summaries_train": {"student_id": "object", "prompt_id": "object", "text": "object", "content": "float64", "wording": "float64"},
    "prompts_train": {"prompt_id": "object", "prompt_question": "object", "prompt_title": "object", "prompt_text": "object"},
}
DATASETS = {"summaries_train": summaries, "prompts_train": prompts}

schema_rows = []
for dataset_name, df in DATASETS.items():
    expected = EXPECTED_SCHEMA[dataset_name]
    assert df.columns.tolist() == list(expected), f"Columnas inesperadas en {dataset_name}"
    for column in df.columns:
        actual_type = str(df[column].dtype)
        expected_type = expected[column]
        schema_rows.append({"dataset": dataset_name, "variable": column, "tipo_observado": actual_type, "tipo_esperado": expected_type, "tipo_válido": actual_type == expected_type, "valores_únicos": df[column].nunique(dropna=False)})

schema_report = pd.DataFrame(schema_rows)
assert schema_report["tipo_válido"].all(), "Se detectaron tipos de datos inesperados"
display(schema_report)

,dataset,variable,tipo_observado,tipo_esperado,tipo_válido,valores_únicos
0,summaries_train,student_id,object,object,True,7165
1,summaries_train,prompt_id,object,object,True,4
2,summaries_train,text,object,object,True,7165
3,summaries_train,content,float64,float64,True,1134
4,summaries_train,wording,float64,float64,True,1134
5,prompts_train,prompt_id,object,object,True,4
6,prompts_train,prompt_question,object,object,True,4
7,prompts_train,prompt_title,object,object,True,4
8,prompts_train,prompt_text,object,object,True,4


Los identificadores y textos se conservan como cadenas; `content` y `wording` son variables numéricas continuas. Sus valores negativos son posibles porque las puntuaciones de la competencia están estandarizadas, por lo que no deben interpretarse como errores de captura.

## 3. Valores faltantes y vacíos

In [3]:
missing_rows = []
for dataset_name, df in DATASETS.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        missing_rows.append({"dataset": dataset_name, "variable": column, "faltantes": missing_count, "porcentaje_faltante": round(100 * missing_count / len(df), 4)})

missing_report = pd.DataFrame(missing_rows)
display(missing_report)
print(f"Total de valores faltantes: {missing_report['faltantes'].sum():,}")

,dataset,variable,faltantes,porcentaje_faltante
0,summaries_train,student_id,0,0.0
1,summaries_train,prompt_id,0,0.0
2,summaries_train,text,0,0.0
3,summaries_train,content,0,0.0
4,summaries_train,wording,0,0.0
5,prompts_train,prompt_id,0,0.0
6,prompts_train,prompt_question,0,0.0
7,prompts_train,prompt_title,0,0.0
8,prompts_train,prompt_text,0,0.0


Total de valores faltantes: 0


In [4]:
blank_rows = []
for dataset_name, df in DATASETS.items():
    for column in df.select_dtypes(include="object").columns:
        text = df[column].fillna("").astype(str)
        blank_rows.append({"dataset": dataset_name, "variable": column, "cadenas_vacías_o_espacios": int(text.str.strip().eq("").sum())})

blank_report = pd.DataFrame(blank_rows)
display(blank_report)

,dataset,variable,cadenas_vacías_o_espacios
0,summaries_train,student_id,0
1,summaries_train,prompt_id,0
2,summaries_train,text,0
3,prompts_train,prompt_id,0
4,prompts_train,prompt_question,0
5,prompts_train,prompt_title,0
6,prompts_train,prompt_text,0


## 4. Duplicados y unicidad de claves

In [5]:
duplicate_report = pd.DataFrame([
    {"control": "Filas completamente duplicadas en summaries", "cantidad": int(summaries.duplicated().sum())},
    {"control": "Filas completamente duplicadas en prompts", "cantidad": int(prompts.duplicated().sum())},
    {"control": "student_id duplicados", "cantidad": int(summaries["student_id"].duplicated().sum())},
    {"control": "Pares student_id + prompt_id duplicados", "cantidad": int(summaries.duplicated(["student_id", "prompt_id"]).sum())},
    {"control": "Textos duplicados", "cantidad": int(summaries["text"].duplicated().sum())},
    {"control": "prompt_id duplicados en prompts", "cantidad": int(prompts["prompt_id"].duplicated().sum())},
])
display(duplicate_report)

,control,cantidad
0,Filas completamente duplicadas en summaries,0
1,Filas completamente duplicadas en prompts,0
2,student_id duplicados,0
3,Pares student_id + prompt_id duplicados,0
4,Textos duplicados,0
5,prompt_id duplicados en prompts,0


## 5. Integridad referencial

Cada `prompt_id` usado por un resumen debe existir exactamente una vez en `prompts_train`. También se comprueba que ningún prompt de entrenamiento quede sin respuestas.

In [6]:
summary_prompt_ids = set(summaries["prompt_id"])
prompt_ids = set(prompts["prompt_id"])
unknown_prompt_ids = sorted(summary_prompt_ids - prompt_ids)
unused_prompt_ids = sorted(prompt_ids - summary_prompt_ids)

referential_report = pd.DataFrame([
    {"control": "prompt_id de summaries inexistentes en prompts", "cantidad": len(unknown_prompt_ids), "detalle": ", ".join(unknown_prompt_ids) or "Ninguno"},
    {"control": "prompt_id sin resúmenes asociados", "cantidad": len(unused_prompt_ids), "detalle": ", ".join(unused_prompt_ids) or "Ninguno"},
])
display(referential_report)

,control,cantidad,detalle
0,prompt_id de summaries inexistentes en prompts,0,Ninguno
1,prompt_id sin resúmenes asociados,0,Ninguno


## 6. Calidad del texto

Se distinguen los problemas de borde (espacios o saltos antes/después del contenido) de los espacios internos. Sólo los primeros se corrigen: colapsar espacios internos podría alterar rasgos lingüísticos relevantes para el EDA.

In [7]:
CONTROL_CHARS = r"[\x00-\x08\x0B\x0C\x0E-\x1F]"
text_quality_rows = []
text_columns = {"summaries_train": ["student_id", "prompt_id", "text"], "prompts_train": ["prompt_id", "prompt_question", "prompt_title", "prompt_text"]}

for dataset_name, columns in text_columns.items():
    df = DATASETS[dataset_name]
    for column in columns:
        values = df[column].fillna("").astype(str)
        text_quality_rows.append({"dataset": dataset_name, "variable": column, "vacíos": int(values.str.strip().eq("").sum()), "espacio_en_bordes": int(values.ne(values.str.strip()).sum()), "caracteres_de_control": int(values.str.contains(CONTROL_CHARS, regex=True).sum()), "longitud_mínima": int(values.str.len().min()), "longitud_mediana": round(float(values.str.len().median()), 1), "longitud_máxima": int(values.str.len().max())})

text_quality_report = pd.DataFrame(text_quality_rows)
display(text_quality_report)

,dataset,variable,vacíos,espacio_en_bordes,caracteres_de_control,longitud_mínima,longitud_mediana,longitud_máxima
0,summaries_train,student_id,0,0,0,12,12.0,12
1,summaries_train,prompt_id,0,0,0,6,6.0,6
2,summaries_train,text,0,2265,0,114,320.0,3940
3,prompts_train,prompt_id,0,0,0,6,6.0,6
4,prompts_train,prompt_question,0,0,0,77,104.5,184
5,prompts_train,prompt_title,0,0,0,10,18.5,25
6,prompts_train,prompt_text,0,0,0,3345,3465.0,5136


In [8]:
summary_text_profile = summaries["text"].str.len().describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).to_frame("caracteres").round(2)
summary_text_profile["palabras"] = summaries["text"].str.split().str.len().describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).round(2)
display(summary_text_profile)

,caracteres,palabras
count,7165.00,7165.00
mean,418.78,74.81
std,307.83,53.50
min,114.00,22.00
1%,133.00,26.00
25%,216.00,39.00
50%,320.00,58.00
75%,513.00,92.00
99%,1572.60,273.36
max,3940.00,647.00


## 7. Variables objetivo

Se validan valores numéricos finitos y se describe su rango. No se eliminan valores extremos: son puntuaciones válidas y su estudio corresponde al EDA de objetivos del notebook `02`.

In [9]:
target_columns = ["content", "wording"]
target_quality = summaries[target_columns].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).T.round(4)
target_quality["no_finitos"] = [int((~np.isfinite(summaries[column])).sum()) for column in target_columns]
display(target_quality)
print("Correlación content-wording: " f"{summaries['content'].corr(summaries['wording']):.4f}")

,count,mean,std,min,1%,25%,50%,75%,99%,max,no_finitos
content,7165.0,-0.0149,1.0436,-1.7299,-1.5472,-0.7995,-0.0938,0.4997,2.9417,3.9003,0
wording,7165.0,-0.0631,1.0360,-1.9626,-1.7955,-0.8727,-0.0818,0.5038,2.7332,4.3107,0


Correlación content-wording: 0.7514


## 8. Cobertura por prompt

In [10]:
prompt_distribution = (summaries.groupby("prompt_id", as_index=False).agg(resúmenes=("student_id", "size"), estudiantes_únicos=("student_id", "nunique")).merge(prompts[["prompt_id", "prompt_title"]], on="prompt_id", how="left", validate="one_to_one"))
prompt_distribution["proporción"] = (prompt_distribution["resúmenes"] / len(summaries)).round(4)
prompt_distribution = prompt_distribution[["prompt_id", "prompt_title", "resúmenes", "estudiantes_únicos", "proporción"]].sort_values("resúmenes", ascending=False, ignore_index=True)
display(prompt_distribution)

,prompt_id,prompt_title,resúmenes,estudiantes_únicos,proporción
0,39c16e,On Tragedy,2057,2057,0.2871
1,3b9047,Egyptian Social Structure,2009,2009,0.2804
2,ebad26,Excerpt from The Jungle,1996,1996,0.2786
3,814d6b,The Third Wave,1103,1103,0.1539


## 9. Matriz de controles y decisiones

In [11]:
boundary_whitespace_count = int(summaries["text"].ne(summaries["text"].str.strip()).sum())

quality_checks = pd.DataFrame([
    {"control": "Esquema y tipos esperados", "estado": "OK", "resultado": "9 variables válidas", "decisión": "Conservar tipos"},
    {"control": "Valores faltantes", "estado": "OK", "resultado": f"{int(missing_report['faltantes'].sum())} valores", "decisión": "No imputar"},
    {"control": "Filas/llaves duplicadas", "estado": "OK", "resultado": f"{int(duplicate_report['cantidad'].sum())} casos", "decisión": "No eliminar filas"},
    {"control": "Integridad de prompt_id", "estado": "OK" if not unknown_prompt_ids else "ERROR", "resultado": f"{len(unknown_prompt_ids)} claves huérfanas", "decisión": "Conservar relación"},
    {"control": "Textos vacíos", "estado": "OK", "resultado": f"{int(blank_report['cadenas_vacías_o_espacios'].sum())} textos", "decisión": "No eliminar filas"},
    {"control": "Espacio en bordes de text", "estado": "CORREGIR" if boundary_whitespace_count else "OK", "resultado": f"{boundary_whitespace_count} textos", "decisión": "Aplicar str.strip() en copia interim"},
    {"control": "Scores no finitos", "estado": "OK", "resultado": f"{int(target_quality['no_finitos'].sum())} valores", "decisión": "Conservar scores"},
])
display(quality_checks)

,control,estado,resultado,decisión
0,Esquema y tipos esperados,OK,9 variables válidas,Conservar tipos
1,Valores faltantes,OK,0 valores,No imputar
2,Filas/llaves duplicadas,OK,0 casos,No eliminar filas
3,Integridad de prompt_id,OK,0 claves huérfanas,Conservar relación
4,Textos vacíos,OK,0 textos,No eliminar filas
5,Espacio en bordes de text,CORREGIR,2265 textos,Aplicar str.strip() en copia interim
6,Scores no finitos,OK,0 valores,Conservar scores


## 10. Limpieza mínima y exportación

Se crean copias para `data/interim/`. La única corrección necesaria es retirar espacios y saltos de línea en los bordes de las columnas textuales; no se altera el contenido interno, las puntuaciones ni el número de filas.

In [12]:
summaries_clean = summaries.copy()
prompts_clean = prompts.copy()

for column in summaries_clean.select_dtypes(include="object").columns:
    summaries_clean[column] = summaries_clean[column].str.strip()
for column in prompts_clean.select_dtypes(include="object").columns:
    prompts_clean[column] = prompts_clean[column].str.strip()

assert summaries_clean.shape == summaries.shape
assert prompts_clean.shape == prompts.shape
assert summaries_clean["student_id"].is_unique
assert prompts_clean["prompt_id"].is_unique
assert set(summaries_clean["prompt_id"]).issubset(set(prompts_clean["prompt_id"]))
assert not summaries_clean["text"].str.strip().eq("").any()
assert summaries_clean["text"].eq(summaries_clean["text"].str.strip()).all()
assert np.isfinite(summaries_clean[target_columns].to_numpy()).all()

summaries_clean.to_csv(INTERIM_DATA_DIR / "summaries_train_clean.csv", index=False)
prompts_clean.to_csv(INTERIM_DATA_DIR / "prompts_train_clean.csv", index=False)

for filename, report in {
    "01_schema_report.csv": schema_report,
    "01_missing_report.csv": missing_report,
    "01_duplicate_report.csv": duplicate_report,
    "01_referential_report.csv": referential_report,
    "01_text_quality_report.csv": text_quality_report,
    "01_target_quality_report.csv": target_quality.reset_index(names="variable"),
    "01_prompt_distribution.csv": prompt_distribution,
    "01_quality_checks.csv": quality_checks,
}.items():
    report.to_csv(TABLES_DIR / filename, index=False)

print("✓ Datos limpios guardados en data/interim/.")
print("✓ Reportes de auditoría guardados en outputs/tables/.")

✓ Datos limpios guardados en data/interim/.
✓ Reportes de auditoría guardados en outputs/tables/.


In [13]:
verification = pd.DataFrame({
    "dataset": ["summaries_train_clean", "prompts_train_clean"],
    "filas_originales": [len(summaries), len(prompts)],
    "filas_limpias": [len(summaries_clean), len(prompts_clean)],
    "faltantes_finales": [int(summaries_clean.isna().sum().sum()), int(prompts_clean.isna().sum().sum())],
    "duplicados_finales": [int(summaries_clean.duplicated().sum()), int(prompts_clean.duplicated().sum())],
})
display(verification)

,dataset,filas_originales,filas_limpias,faltantes_finales,duplicados_finales
0,summaries_train_clean,7165,7165,0,0
1,prompts_train_clean,4,4,0,0


## Conclusiones

- Los datos de entrenamiento contienen **7,165 resúmenes** y **4 prompts**, con los tipos y columnas esperados.
- No hay valores faltantes, cadenas vacías, filas duplicadas, `student_id` repetidos, textos duplicados ni `prompt_id` huérfanos.
- Los cuatro prompts tienen respuestas, aunque la cantidad por prompt no es uniforme; esta diferencia debe considerarse al comparar grupos en el EDA.
- Se detectaron **2,265 resúmenes con espacio o saltos de línea en los bordes**. Se corrigieron únicamente en la copia `data/interim/summaries_train_clean.csv`.
- Las puntuaciones `content` y `wording` son finitas. Los valores negativos y extremos se conservan porque forman parte de la escala estandarizada y deben analizarse, no eliminarse automáticamente.
- Después de la limpieza se preserva el 100 % de las observaciones y la integridad de las claves. Los datos quedan listos para `02_eda_targets.ipynb`.